<a href="https://www.kaggle.com/code/kawsarahmedkazol/backend-bangla-voice-ai-api?scriptVersionId=312643273" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🎙️ Bangla Voice AI API

Simple backend for Bangla voice chat: Speech → Text → AI → Voice

## 🔗 API Base URL
Your Ngrok URL (changes on restart):After Run this notebook you will get this type url
> https://your-ngrok-url.ngrok-free.dev(Example)

## How to use 
- add the Huggingface token && ngrok token in the kaggle addons
- Run the Notebook
- paste the ngrok public url to this website [Bangla Voice chat ai](https://bangla-voice-chat-ai.onrender.com/)
- This will also give you swagger api docs 


## 👤 Author
Kawsar Ahmed Kazol | https://github.com/kazol196295

In [1]:

!apt-get update -qq
!apt-get install -qq libsndfile1 ffmpeg

!pip install -U pip setuptools wheel cython ninja

!pip install  accelerate peft

!pip install  bitsandbytes

!pip install soundfile librosa numpy scipy
!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart huggingface-hub

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.2 MB/s eta 0:00:00:00:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Attempting uninstall: cython
    Found existing installation: Cython 3.0.12
    Uninstalling Cython-3.0.12:
      Successfully uninstalled Cython-3.0.12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 51.4 MB/s  0:00:01m0:00:0100:01


In [3]:
!pip install coqui-tts -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
# import TTS
# print("Coqui TTS version:", TTS.__version__)
# from TTS.api import TTS
# print("Successfully imported TTS.api")
from platform import python_version
print(python_version())


3.12.12


In [6]:
!pip show transformers coqui-tts torch

Name: transformers
Version: 5.0.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer-slim
Required-by: coqui-tts, kaggle-environments, peft, sentence-transformers
---
Name: coqui-tts
Version: 0.27.5
Summary: Deep learning for Text to Speech.
Home-page: https://github.com/idiap/coqui-ai-TTS
Author: 
Author-email: Eren Gölge <egolge@coqui.ai>
License: MPL-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyascii, coqpit-c

In [ ]:
import os
import time
import torch
import base64
import uvicorn
import tempfile
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok, conf
from huggingface_hub import login, hf_hub_download, list_repo_files
from transformers import pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from fastapi import File, UploadFile

# ==========================================================
# EXTENDED MONKEY-PATCH for coqui-tts + transformers + numpy
# ==========================================================
import transformers.utils.import_utils as _iu
from packaging.version import Version
import transformers.pytorch_utils as _pu
import torch

if not hasattr(_iu, 'is_torch_greater_or_equal'):
    _iu.is_torch_greater_or_equal = lambda v: Version(torch.__version__) >= Version(v)
if not hasattr(_iu, 'is_torch_available'):
    _iu.is_torch_available = lambda: torch is not None

# Fix torchcodec (new in transformers 5.x)
if not hasattr(_iu, 'is_torchcodec_available'):
    _iu.is_torchcodec_available = lambda: False

# Fix isin_mps_friendly (old Tortoise legacy code)
if not hasattr(_pu, 'isin_mps_friendly'):
    _pu.isin_mps_friendly = lambda x, y: torch.isin(x, y)

# NOW import TTS
from TTS.api import TTS
# ==========================================================

# ==========================================================
# 1. CONFIGURATION
# ==========================================================

from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")

if not HF_TOKEN or not NGROK_TOKEN:
    raise ValueError("❌ Missing tokens! Add HF_TOKEN & NGROK_TOKEN in Kaggle Add-ons → Secrets")

login(token=HF_TOKEN)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

# ==========================================================
# 2. API TAGS
# ==========================================================

tags_metadata = [
    {"name": "Speech Recognition", "description": "Convert Bangla speech audio to text."},
    {"name": "Chat AI", "description": "Generate Bangla AI responses using Llama 3.1."},
    {"name": "Text To Speech", "description": "Convert Bangla text to speech."}
]

# ==========================================================
# 3. FASTAPI INITIALIZATION
# ==========================================================

app = FastAPI(
    title="Unified Bangla Voice AI API",
    description="""
🎙️ End-to-End Bangla Voice AI System

Pipeline: Speech → Text → AI Response → Voice

Models:
• Whisper Bengali ASR
• Llama 3.1 8B Instruct
• Bangla VITS TTS
""",
    version="1.0.0",
    contact={"name": "Kawsar Ahmed Kazol", "url": "https://github.com/kazol196295"},
    openapi_tags=tags_metadata,
    docs_url="/docs",
    redoc_url="/redoc"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ==========================================================
# 4. MODEL HOLDERS
# ==========================================================

models = {
    "asr_processor": None,
    "asr_model": None,
    "llm": None,
    "tokenizer": None,
    "tts": None,
    "loaded": False
}
model_pipeline = None
# ==========================================================
# 5. MODEL LOADING
# ==========================================================

@app.on_event("startup")
async def load_all_models():
    global models
    print(f"🔄 Loading all models into {DEVICE} ...")

    try:
        # =========================================
        # 🎙️ LOAD ASR (Your Bengali Whisper)
        # =========================================
        print("⏳ Loading Whisper Bengali ASR...")
        ASR_MODEL_ID = "kazol196295/whisper-bengali-final-1.3"
        
        models["asr_processor"] = AutoProcessor.from_pretrained(ASR_MODEL_ID)
        models["asr_model"] = AutoModelForSpeechSeq2Seq.from_pretrained(
            ASR_MODEL_ID,
            torch_dtype=TORCH_DTYPE,
        ).to(DEVICE)
        print("✅ ASR Loaded!")

        # =========================================
        # 🧠 LOAD LLM (Llama 3.1 8B) — unchanged
        # =========================================
        print("⏳ Loading Llama 3.1 8B...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        llm_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        models["tokenizer"] = AutoTokenizer.from_pretrained(llm_id)
        if models["tokenizer"].pad_token_id is None:
            models["tokenizer"].pad_token_id = models["tokenizer"].eos_token_id

        models["llm"] = AutoModelForCausalLM.from_pretrained(
            llm_id,
            quantization_config=bnb_config,
            device_map="auto"
        )
        print("✅ Llama 3.1 Loaded!")

        # =========================================
        # 🔊 LOAD TTS (Bangla VITS) 
        # =========================================
        print("⏳ Loading Bangla VITS TTS...")
        repo_id = "kazol196295/bangla-vits-female-1.2"
        files = list_repo_files(repo_id)
        pth_files = [f for f in files if f.endswith(".pth") and f.startswith("current_model/")]
        latest_model = sorted(pth_files)[-1]

        config_path = hf_hub_download(repo_id=repo_id, filename="current_model/config.json")
        model_path = hf_hub_download(repo_id=repo_id, filename=latest_model)

        models["tts"] = TTS(
            model_path=model_path,
            config_path=config_path,
            gpu=torch.cuda.is_available()
        )
        print("✅ TTS Loaded!")

        models["loaded"] = True
        print("🎉 ALL MODELS LOADED SUCCESSFULLY!")

    except Exception as e:
        print(f"❌ Initialization Error: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()

# ==========================================================
# 6. API SCHEMAS
# ==========================================================

class AudioPayload(BaseModel):
    audio_base64: str

class TextPayload(BaseModel):
    text: str
    history: list = []

# ==========================================================
# 7. SPEECH TO TEXT API
# ==========================================================



import librosa
import soundfile as sf
import traceback
import tempfile
import os

@app.post("/transcribe")
async def transcribe_audio(file: UploadFile = File(...)):
    if not models["loaded"] or models["asr_model"] is None:
        raise HTTPException(status_code=503, detail="Models still loading...")

    temp_filename = None
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
            tmp.write(await file.read())
            temp_filename = tmp.name

        # Load + resample to 16kHz
        audio, sr = librosa.load(temp_filename, sr=16000)

        # Manually run feature extraction
        processor = models["asr_processor"]
        model     = models["asr_model"]

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        )
        input_features = inputs.input_features.to(DEVICE, dtype=TORCH_DTYPE)

        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                language="bn",      # Bengali
                task="transcribe",
                max_new_tokens=444,
            )

        transcription = processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True
        )[0]

        return {"transcription": transcription.strip()}

    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {str(e)}")
    finally:
        if temp_filename and os.path.exists(temp_filename):
            os.remove(temp_filename)
# ==========================================================
# 8. CHAT API
# ==========================================================

@app.post("/api/chat", tags=["Chat AI"])
async def chat(payload: TextPayload):
    if not models["loaded"]:
        raise HTTPException(status_code=503, detail="Models Loading")

    try:
        messages = [
            {"role": "system", "content": "Always answer shortly in Bangla."}
        ]

        for msg in payload.history:
            messages.append(msg)

        messages.append({"role": "user", "content": payload.text})

        prompt = models["tokenizer"].apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = models["tokenizer"](prompt, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            outputs = models["llm"].generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.6,
                top_p=0.9,
                do_sample=True,
                pad_token_id=models["tokenizer"].eos_token_id
            )

        input_length = inputs["input_ids"].shape[1]
        new_tokens = outputs[0][input_length:]
        response = models["tokenizer"].decode(new_tokens, skip_special_tokens=True)

        return {"text": response.strip()}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ==========================================================
# 9. TEXT TO SPEECH API
# ==========================================================

@app.post("/api/synthesize", tags=["Text To Speech"])
async def synthesize(payload: TextPayload):
    if not models["loaded"]:
        raise HTTPException(status_code=503, detail="Models Loading")

    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
            output_path = tmp.name

        models["tts"].tts_to_file(
            text=payload.text,
            file_path=output_path
        )

        with open(output_path, "rb") as f:
            audio_base64 = base64.b64encode(f.read()).decode("utf-8")

        os.unlink(output_path)

        return {"audio_base64": audio_base64}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ==========================================================
# 10. SERVER START
# ==========================================================

if __name__ == "__main__":
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()

    public_url = ngrok.connect(8000).public_url

    print("\n" + "=" * 60)
    print("🚀 API LIVE")
    print("URL:", public_url)
    print("Swagger:", public_url + "/docs")
    print("=" * 60)

    nest_asyncio.apply()
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)

    import asyncio
    asyncio.get_event_loop().run_until_complete(server.serve())

/tmp/ipykernel_55/1662923221.py:119: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


INFO:     Started server process [55]
INFO:     Waiting for application startup.



🚀 API LIVE
URL: https://peremptory-fourthly-argelia.ngrok-free.dev
Swagger: https://peremptory-fourthly-argelia.ngrok-free.dev/docs
🔄 Loading all models into cuda:0 ...
⏳ Loading Whisper Bengali ASR...


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

✅ TTS Loaded!
🎉 ALL MODELS LOADED SUCCESSFULLY!


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=444) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'trans

INFO:     74.220.52.251:0 - "POST /transcribe HTTP/1.1" 200 OK
INFO:     74.220.52.251:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     74.220.52.251:0 - "POST /api/synthesize HTTP/1.1" 200 OK


Both `max_new_tokens` (=444) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     74.220.52.251:0 - "POST /transcribe HTTP/1.1" 200 OK
INFO:     74.220.52.251:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     74.220.52.251:0 - "POST /api/synthesize HTTP/1.1" 200 OK
